# Pipeline Vigil Energy — Portfólio de Manutenção Preditiva

### Considerações
Claro! Vou explicar cada um desses conceitos como se você nunca tivesse ouvido falar deles antes, usando uma analogia simples.

## 1. **Baseline saudável com ciclos diários de carga**

Imagine que você tem um **restaurante**. Um “baseline saudável” seria o funcionamento normal do restaurante em um dia típico: os chefes cozinham, os garçons atendem, os clientes chegam e saem. Mas nem todo dia é igual: no almoço (12h) e no jantar (19h) há **picos de movimento** – são os “ciclos diários de carga”. Durante o dia, a carga vai subindo e descendo de forma previsível.

No mundo da tecnologia (servidores, bancos de dados, redes), “baseline saudável com ciclos diários de carga” significa:  
> *“Sabemos como o sistema se comporta em um dia normal, com seus horários de pico e de baixa atividade. Esse é nosso padrão de referência para detectar problemas.”*

## 2. **Cenário de falha na T-04**

Agora, suponha que no **tempo T-04** (4 minutos antes de um evento importante, como uma grande promoção no restaurante), acontece uma **falha** – por exemplo, o forno queima.

“Cenário de falha na T-04” é um **teste planejado** onde os engenheiros forçam uma pane no sistema **pouco antes de um momento crítico** (como 4 minutos antes do horário de pico). O objetivo é ver se o sistema consegue se recuperar rápido o suficiente para não prejudicar os clientes.

> *“Vamos quebrar algo propositalmente em um momento delicado para treinar a equipe e ver se os mecanismos de emergência funcionam.”*

## 3. **Anomalia temporária na T-02**

Aqui, no **tempo T-02** (2 minutos antes do evento importante), acontece algo estranho mas passageiro. Voltando ao restaurante: seria como se todos os pedidos começassem a sair com o nome do cliente errado por 30 segundos, mas depois voltassem ao normal sozinhos.

Na tecnologia, “anomalia temporária na T-02” significa:  
> *“Vamos introduzir um comportamento esquisito (como lentidão, erros aleatórios ou perda de pacotes de rede) por poucos instantes, bem perto do horário de pico, para ver se o sistema percebe, se corrige sozinho e se o impacto nos usuários é mínimo.”*

---

## Resumo da história (aplicação real)

Se você é responsável por um sistema de vendas online:

- O **baseline saudável** seria: “Entre 10h e 12h temos 100 pedidos/minuto; entre 12h e 14h, 500 pedidos/minuto. A resposta do site é de 0,5 segundo.”
- O **cenário de falha na T-04** seria: “Vamos desligar um dos bancos de dados 4 minutos antes das 12h e ver se o site continua no ar.”
- A **anomalia temporária na T-02** seria: “Vamos atrasar artificialmente as respostas do servidor por 10 segundos, 2 minutos antes das 12h, e ver se o sistema de recuperação automática funciona.”

Esses testes ajudam a garantir que o sistema não vai quebrar justo na hora do maior movimento.


## 0. Preparação de ambiente

In [1]:
# Criação de diretórios necessários
import os
#os.makedirs('dados', exist_ok=True)
#os.makedirs('outputs', exist_ok=True)
#from pathlib import Path
#print('Estrutura de pastas pronta!')

In [2]:
# Importar bibliotecas primeiro
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

In [3]:
# Definindo o diretório onde os dados estão armazenados
# Montagem do drive do Google Colab
from google.colab import drive
drive.mount('/content/drive')

# Defina o caminho do diretório Exemplo para Google Drive: '/content/drive/MyDrive/Sua_Pasta'
caminho_diretorio = '/content/drive/MyDrive/Colab Notebooks/Portifólio de Projetos/vigil_artefatos/dados/'

# Verifica se o diretório existe para evitar erros
if os.path.exists(caminho_diretorio):
    arquivos = os.listdir(caminho_diretorio)
    print(f"Arquivos encontrados na pasta '{caminho_diretorio}':")
    for arquivo in arquivos:
        print(f"- {arquivo}")
else:
    print("O diretório especificado não foi encontrado. Verifique o caminho.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Arquivos encontrados na pasta '/content/drive/MyDrive/Colab Notebooks/Portifólio de Projetos/vigil_artefatos/dados/':
- dados_sensores_vigil.csv
- janelas_falha_ground_truth.csv


## 1. Geração de Dados Sintéticos de Sensores IoT
Gera um baseline saudável com ciclos diários de carga e adiciona os cenários de falha na T-04 e anomalia temporária na T-02.

###DADOS ALEATÓRIOS APENAS PARA ESTUDO DE CASO

Esse código criou um "laboratório virtual" com dados de 6 turbinas (como as de usina eólica ou hidrelétrica). São 12.096 medições registradas ao longo de 14 dias, com uma leitura a cada 10 minutos.

Medindo:
Vibração (tremor)
Temperatura (febre)
Pressão (tensão interna)
Corrente elétrica (energia consumida)

O código adiciona um ciclo natural parecido com uma onda: durante o dia a turbina trabalha mais, à noite menos. É como sua respiração: sobe e desce num ritmo diário. Isso deixa os dados realistas, não uma linha reta chata.

Os cenários de problema (o que você pediu antes)
O código injetou propositalmente dois defeitos, exatamente como você mencionou:

Falha na T-04 (cenário de falha)
O que acontece: Nas últimas 8 horas dos 14 dias, a turbina T-04 começa a desenvolver um problema no rolamento (peça que gira). A vibração e temperatura sobem cada vez mais rápido (crescimento exponencial).

Por que é uma "falha": É um problema que piora com o tempo. Se ninguém agir, a turbina vai quebrar.

Anomalia temporária na T-02 (transiente)
O que acontece: Entre o 9º e o 11º dia, a temperatura da T-02 sobe 7°C durante 2 horas e depois volta ao normal sozinha.

Por que é "anomalia temporária": É um susto passageiro – como uma febre que vai embora sem remédio. Pode ser um filtro entupido que desentope sozinho.

Os dados podem ser usados para treinar inteligência artificial a detectar esses problemas antes que virem uma quebra de verdade. É como um simulador de voo para engenheiros de manutenção.

In [4]:
"""
Vigil Energy — 01 · Gerador de dados sintéticos de sensores IoT
====================================================================================
Gera uma base de telemetria realista para a frota de turbinas monitoradas.
"""

from __future__ import annotations

SEED = 42
rng = np.random.default_rng(SEED)

ATIVOS = ["T-01", "T-02", "T-03", "T-04", "T-05", "T-06"]
DIAS = 14
RESOLUCAO_MIN = 10
PASSOS_POR_DIA = 24 * 60 // RESOLUCAO_MIN
N = DIAS * PASSOS_POR_DIA
INICIO = datetime(2026, 3, 1, 0, 0, 0)

BASELINE = {
    "T-01": dict(vib=2.0, temp=62, pres=4.2, cur=118),
    "T-02": dict(vib=2.3, temp=65, pres=4.0, cur=121),
    "T-03": dict(vib=1.8, temp=60, pres=4.3, cur=115),
    "T-04": dict(vib=2.1, temp=63, pres=4.1, cur=120),
    "T-05": dict(vib=2.6, temp=67, pres=3.9, cur=124),
    "T-06": dict(vib=1.9, temp=61, pres=4.2, cur=117),
}

def serie_normal(base: dict, n: int) -> dict[str, np.ndarray]:
    t = np.arange(n)
    ciclo = np.sin(2 * np.pi * t / PASSOS_POR_DIA)
    vib = base["vib"] + 0.25 * ciclo + rng.normal(0, 0.18, n)
    temp = base["temp"] + 2.5 * ciclo + rng.normal(0, 0.8, n)
    pres = base["pres"] + 0.10 * ciclo + rng.normal(0, 0.06, n)
    cur = base["cur"] + 3.0 * ciclo + rng.normal(0, 1.1, n)
    return {"vibracao_mm_s": vib, "temperatura_C": temp, "pressao_bar": pres, "corrente_A": cur}

def injeta_falha_rolamento(series: dict[str, np.ndarray], n: int):
    passos_8h = 8 * 60 // RESOLUCAO_MIN
    ini = n - passos_8h
    ramp = np.linspace(0, 1, passos_8h)
    series["vibracao_mm_s"][ini:] += np.expm1(1.6 * ramp) * 1.52
    series["temperatura_C"][ini:] += np.expm1(1.6 * ramp) * 1.4
    series["corrente_A"][ini:] += np.expm1(1.6 * ramp) * 1.0
    return ini

def injeta_transiente_temp(series: dict[str, np.ndarray], n: int):
    ini = 9 * PASSOS_POR_DIA
    passos_2h = 2 * 60 // RESOLUCAO_MIN
    fim = ini + passos_2h
    bump = np.sin(np.linspace(0, np.pi, passos_2h)) * 7.0
    series["temperatura_C"][ini:fim] += bump
    series["corrente_A"][ini:fim] += bump * 0.3
    return ini, fim

def main():
    # Usa o diretório especificado
    dados_dir = caminho_diretorio
    os.makedirs(dados_dir, exist_ok=True)

    timestamps = [INICIO + timedelta(minutes=RESOLUCAO_MIN * i) for i in range(N)]
    linhas = []
    ground_truth = []

    for ativo in ATIVOS:
        s = serie_normal(BASELINE[ativo], N)

        if ativo == "T-04":
            ini = injeta_falha_rolamento(s, N)
            ground_truth.append(dict(ativo=ativo, tipo="falha_rolamento",
                                     inicio=timestamps[ini].isoformat(),
                                     fim=timestamps[-1].isoformat(),
                                     critico=True))
        if ativo == "T-02":
            ini, fim = injeta_transiente_temp(s, N)
            ground_truth.append(dict(ativo=ativo, tipo="transiente_temp",
                                     inicio=timestamps[ini].isoformat(),
                                     fim=timestamps[fim - 1].isoformat(),
                                     critico=False))

        for i in range(N):
            linhas.append({
                "timestamp": timestamps[i].isoformat(),
                "ativo": ativo,
                "vibracao_mm_s": round(float(s["vibracao_mm_s"][i]), 4),
                "temperatura_C": round(float(s["temperatura_C"][i]), 3),
                "pressao_bar": round(float(s["pressao_bar"][i]), 4),
                "corrente_A": round(float(s["corrente_A"][i]), 3),
            })

    df = pd.DataFrame(linhas)
    gt = pd.DataFrame(ground_truth)

    out_csv = os.path.join(dados_dir, "dados_sensores_vigil.csv")
    gt_csv = os.path.join(dados_dir, "janelas_falha_ground_truth.csv")
    df.to_csv(out_csv, index=False)
    gt.to_csv(gt_csv, index=False)

    print(f"OK  {len(df):,} leituras geradas para {len(ATIVOS)} ativos.")
    print(f"    -> {out_csv}")
    print(f"    -> {gt_csv}")

if __name__ == "__main__":
    main()

OK  12,096 leituras geradas para 6 ativos.
    -> /content/drive/MyDrive/Colab Notebooks/Portifólio de Projetos/vigil_artefatos/dados/dados_sensores_vigil.csv
    -> /content/drive/MyDrive/Colab Notebooks/Portifólio de Projetos/vigil_artefatos/dados/janelas_falha_ground_truth.csv


In [5]:
# Carregar os DataFrames do diretório especificado
df_sensores = pd.read_csv(os.path.join(caminho_diretorio, 'dados_sensores_vigil.csv'))
df_ground_truth = pd.read_csv(os.path.join(caminho_diretorio, 'janelas_falha_ground_truth.csv'))

print("=" * 50)
print("✅ DADOS CARREGADOS COM SUCESSO!")
print("=" * 50)
print(f"📊 Total de leituras: {len(df_sensores):,}")
print(f"🏭 Turbinas monitoradas: {df_sensores['ativo'].unique()}")
print(f"📅 Período: de {df_sensores['timestamp'].min()} até {df_sensores['timestamp'].max()}")
print("=" * 50)

# Ver as primeiras linhas
print("\n📋 PRIMEIRAS LINHAS DOS SENSORES:")
print(df_sensores.head())

print("\n📋 GROUND TRUTH (GABARITO DAS FALHAS):")
print(df_ground_truth)

✅ DADOS CARREGADOS COM SUCESSO!
📊 Total de leituras: 12,096
🏭 Turbinas monitoradas: ['T-01' 'T-02' 'T-03' 'T-04' 'T-05' 'T-06']
📅 Período: de 2026-03-01T00:00:00 até 2026-03-14T23:50:00

📋 PRIMEIRAS LINHAS DOS SENSORES:
             timestamp ativo  vibracao_mm_s  temperatura_C  pressao_bar  \
0  2026-03-01T00:00:00  T-01         2.0548         61.978       4.2352   
1  2026-03-01T00:10:00  T-01         1.8237         61.581       4.2401   
2  2026-03-01T00:20:00  T-01         2.1569         62.683       4.1652   
3  2026-03-01T00:30:00  T-01         2.2019         62.708       4.2228   
4  2026-03-01T00:40:00  T-01         1.6922         62.446       4.3493   

   corrente_A  
0     118.914  
1     117.957  
2     117.456  
3     117.950  
4     117.204  

📋 GROUND TRUTH (GABARITO DAS FALHAS):
  ativo             tipo               inicio                  fim  critico
0  T-02  transiente_temp  2026-03-10T00:00:00  2026-03-10T01:50:00    False
1  T-04  falha_rolamento  2026-03-14T16:00

## 2. Vigil Detect — Motor Preditivo Multifatorial

Este código implementa um sistema de detecção de falhas preditiva para equipamentos industriais, combinando duas técnicas de machine learning.

Compomentes de avaliação: aplicação do Isolation Forest + LSTM Autoencoder por turbina para capturar desvios de assinatura operativa.
Isolation Forest (Anomalias pontuais e abruptas) é como um detetive que isola comportamentos estranhos rapidamente
O LSTM Autoencoder (Degradação progressiva) é como um copiador perfeito que estranha quando não consegue copiar um comportamento
Juntos, eles aprendem o "jeito normal" de cada turbina e acusam o alerta quando algo foge desse padrão.

Regras:
Monitora 4 variáveis críticas de equipamentos rotativos
Janela deslizante de 12 pontos para capturar padrões temporais
70% dos dados para treino, 30% para detecção

Primeira estratégia - Isolation Forest (Peso 45%)
Função: Detecta outliers baseado em isolamento de pontos
Vantagem: Bom para anomalias pontuais e dados de alta dimensão
Peso menor (0.45) pois é menos sensível a padrões sequenciais

Segunda estratágia - LSTM Autoencoder (Peso 55%)
Função: Aprende padrões temporais normais
Funcionamento: Tenta reconstruir sequências; erro alto = anomalia
Peso maior (0.55) por capturar melhor degradação progressiva

Vantagens
Normaliza as pontuações (z-score - média = 0, desvio = 1), escala diferentes Isolation Forest gera scores que tipicamente variam entre -0,5 e 0,5

LSTM Autoencoder gera erros de reconstrução que variam entre 0 e 100
Combina ambas em um score final ponderado
Mais robusto que usar apenas um método

Limiar Adaptativo
Define threshold baseado nos dados de treino
Apenas os 0.2% mais extremos são considerados anomalias

Sistema Anti-Falso Alarme
Evita disparos esporádicos
Requer confirmação temporal antes de gerar alerta



In [6]:
"""
Vigil Energy — 02 · Vigil Detect (Motor Preditivo - Compatível Colab/Local)
=========================================================================
Detecta anomalias combinando Isolation Forest e LSTM Autoencoder por ativo.
"""

from __future__ import annotations
import os, json, warnings
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
import tensorflow as tf
from tensorflow.keras import layers, Model

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

FEATURES = ["vibracao_mm_s", "temperatura_C", "pressao_bar", "corrente_A"]
JANELA = 12
FRACAO_TREINO = 0.70
DEBOUNCE = 2
QUANTIL_LIMIAR = 0.998
RESOLUCAO_MIN = 10
PESO_IF, PESO_LSTM = 0.45, 0.55

def build_lstm_autoencoder(n_features: int, janela: int) -> Model:
    inp = layers.Input(shape=(janela, n_features))
    x = layers.LSTM(32, activation="tanh", return_sequences=False)(inp)
    x = layers.RepeatVector(janela)(x)
    x = layers.LSTM(32, activation="tanh", return_sequences=True)(x)
    out = layers.TimeDistributed(layers.Dense(n_features))(x)
    model = Model(inp, out)
    model.compile(optimizer="adam", loss="mse")
    return model

def janelas_deslizantes(arr: np.ndarray, janela: int) -> np.ndarray:
    return np.stack([arr[i:i + janela] for i in range(len(arr) - janela + 1)])

def processa_ativo(df_ativo: pd.DataFrame) -> pd.DataFrame:
    df_ativo = df_ativo.sort_values("timestamp").reset_index(drop=True)
    n = len(df_ativo)
    corte = int(n * FRACAO_TREINO)

    X = df_ativo[FEATURES].values.astype("float32")
    scaler = StandardScaler().fit(X[:corte])
    Xs = scaler.transform(X)

    iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=SEED).fit(Xs[:corte])
    if_score = -iso.score_samples(Xs)

    Xw = janelas_deslizantes(Xs, JANELA)
    corte_w = max(1, corte - JANELA + 1)
    ae = build_lstm_autoencoder(len(FEATURES), JANELA)
    ae.fit(Xw[:corte_w], Xw[:corte_w], epochs=12, batch_size=64, verbose=0)
    recon = ae.predict(Xw, verbose=0)
    err = np.mean((Xw - recon) ** 2, axis=(1, 2))
    lstm_score = np.concatenate([np.full(JANELA - 1, err[0]), err])

    def z_treino(v):
        mu, sd = float(np.mean(v[:corte])), float(np.std(v[:corte]) + 1e-9)
        return (v - mu) / sd
    comb = PESO_IF * z_treino(if_score) + PESO_LSTM * z_treino(lstm_score)

    limiar = float(np.quantile(comb[:corte], QUANTIL_LIMIAR))
    score01 = 1.0 / (1.0 + np.exp(-(comb - limiar)))

    df_ativo = df_ativo.copy()
    df_ativo["score_anomalia"] = np.round(score01, 4)
    df_ativo["acima_limiar"] = comb > limiar

    alerta = np.zeros(n, dtype=bool)
    run = 0
    for i in range(n):
        run = run + 1 if df_ativo["acima_limiar"].iloc[i] else 0
        if run >= DEBOUNCE:
            alerta[i] = True
    df_ativo["alerta"] = alerta
    df_ativo["limiar"] = round(limiar, 4)
    return df_ativo

def ponto_anomalo(ts, ativo, gt) -> bool:
    sub = gt[gt["ativo"] == ativo]
    return any((ts >= r.inicio) and (ts <= r.fim) for r in sub.itertuples())

def gera_grafico_t04(t04: pd.DataFrame, falha_ts, out_dir: str):
    import matplotlib
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates

    t04 = t04.sort_values("timestamp")
    fig, ax = plt.subplots(figsize=(11, 4.2), dpi=150)
    fig.patch.set_facecolor("#0D1117"); ax.set_facecolor("#161B22")

    normal = t04[~t04["verdade_anomalia"]]
    anom = t04[t04["verdade_anomalia"]]
    ax.plot(normal["timestamp"], normal["vibracao_mm_s"], color="#3DD68C", lw=1.6, label="Operação normal")
    ax.plot(anom["timestamp"], anom["vibracao_mm_s"], color="#FF4B4B", lw=2.2, label="Degradação de rolamento")

    alertou = t04[t04["alerta"]]
    if len(alertou):
        primeiro = alertou["timestamp"].iloc[0]
        ax.axvline(primeiro, color="#F0A500", ls="--", lw=1.6)
        ax.text(primeiro, ax.get_ylim()[1]*0.92, "  1º alerta", color="#F0A500", fontsize=9, fontfamily="monospace")

    for sp in ax.spines.values(): sp.set_color("#2A3444")
    ax.tick_params(colors="#768BA4", labelsize=8)
    ax.set_ylabel("Vibração (mm/s)", color="#E6EDF3", fontsize=10)
    ax.set_title("Vigil Detect — Turbina T-04 · detecção de falha de rolamento", color="#E6EDF3", fontsize=12, loc="left", pad=12)
    ax.grid(color="#2A3444", alpha=0.5, lw=0.6)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
    ax.legend(facecolor="#1C2330", edgecolor="#2A3444", fontsize=8.5, labelcolor="#E6EDF3", loc="upper left")
    fig.tight_layout()
    path = os.path.join(out_dir, "grafico_anomalia_T04.png")
    fig.savefig(path, facecolor=fig.get_facecolor())
    plt.close(fig)

def main():
    # AJUSTE DEFINITIVO DE ROTAS: Utilizando o caminho base do seu projeto no Drive
    base_dir = '/content/drive/MyDrive/Colab Notebooks/Portifólio de Projetos/vigil_artefatos/'
    dados = os.path.join(base_dir, 'dados')
    out = os.path.join(base_dir, 'outputs')

    # Cria a pasta de outputs dentro do seu Drive, caso ela ainda não exista
    os.makedirs(out, exist_ok=True)

    df = pd.read_csv(os.path.join(dados, "dados_sensores_vigil.csv"))

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    gt = pd.read_csv(os.path.join(dados, "janelas_falha_ground_truth.csv"))
    gt["inicio"] = pd.to_datetime(gt["inicio"])
    gt["fim"] = pd.to_datetime(gt["fim"])

    print("Treinando Vigil Detect por ativo...")
    resultados = []
    for ativo, g in df.groupby("ativo"):
        print(f"  · {ativo} ...")
        resultados.append(processa_ativo(g))
    scored = pd.concat(resultados, ignore_index=True)

    scored["verdade_anomalia"] = [ponto_anomalo(ts, a, gt) for ts, a in zip(scored["timestamp"], scored["ativo"])]
    alertas = scored[scored["alerta"]].copy()

    alertas["severidade"] = np.where(alertas["score_anomalia"] >= 0.80, "CRITICO",
                              np.where(alertas["score_anomalia"] >= 0.55, "ALTO", "MEDIO"))

    t04 = scored[scored["ativo"] == "T-04"]
    falha_ts = gt[gt["ativo"] == "T-04"]["fim"].iloc[0]

    cols = ["timestamp", "ativo", "score_anomalia", "severidade", "verdade_anomalia"] + FEATURES
    alertas[cols].to_csv(os.path.join(out, "alertas_detectados.csv"), index=False)
    scored[["timestamp", "ativo", "score_anomalia", "alerta", "verdade_anomalia"]].to_csv(os.path.join(out, "scores_por_ativo.csv"), index=False)

    gera_grafico_t04(t04, falha_ts, out)
    print("Modelo processado com sucesso! Arquivos salvos na pasta 'outputs' do seu Drive.")

if __name__ == "__main__":
    main()

Treinando Vigil Detect por ativo...
  · T-01 ...
  · T-02 ...
  · T-03 ...
  · T-04 ...
  · T-05 ...
  · T-06 ...
Modelo processado com sucesso! Arquivos salvos na pasta 'outputs' do seu Drive.


In [7]:
"""
Vigil Energy — 03 · Vigil Dashboard (Visualização Sequencial)
========================================================================
Gera a análise visual do motor preditivo em blocos separados e robustos,
garantindo compatibilidade nativa com o Google Colab.
"""

import os
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# Garante que o Plotly renderize corretamente no output padrão do Colab
pio.renderers.default = 'colab'

def gerar_visualizacoes_separadas(out_dir: str):
    scores_file = os.path.join(out_dir, "scores_por_ativo.csv")
    alertas_file = os.path.join(out_dir, "alertas_detectados.csv")

    if not os.path.exists(scores_file):
        print("⚠️ Arquivo de scores não encontrado. Execute o modelo Vigil Detect primeiro.")
        return

    # 1. Carga dos Dados
    df_scores = pd.read_csv(scores_file)
    df_scores['timestamp'] = pd.to_datetime(df_scores['timestamp'])

    df_alertas = pd.DataFrame()
    if os.path.exists(alertas_file):
        df_alertas = pd.read_csv(alertas_file)
        df_alertas['timestamp'] = pd.to_datetime(df_alertas['timestamp'])

    # Paleta de cores padronizada
    cores_turbinas = {
        "T-01": "#1f77b4", "T-02": "#9467bd", "T-03": "#2ca02c",
        "T-04": "#FF4B4B", "T-05": "#e377c2", "T-06": "#8c564b"
    }
    cores_sev = {'CRITICO': '#FF4B4B', 'ALTO': '#F0A500', 'MEDIO': '#3DD68C'}

    print("📊 GERANDO RELATÓRIO VISUAL VIGIL ENERGY...\n")

    # =====================================================================
    # BLOCO 1: Visão Geral - Evolução do Score de Anomalia (Frota Completa)
    # =====================================================================
    fig1 = go.Figure()

    for ativo in sorted(df_scores['ativo'].unique()):
        df_plot = df_scores[df_scores['ativo'] == ativo]

        # Destaca a turbina T-04 (que sabemos ter a falha exponencial)
        width = 2.5 if ativo == "T-04" else 1.0
        opacity = 1.0 if ativo == "T-04" else 0.6

        fig1.add_trace(
            go.Scatter(
                x=df_plot['timestamp'], y=df_plot['score_anomalia'],
                name=ativo, mode='lines',
                line=dict(color=cores_turbinas.get(ativo, "#ffffff"), width=width),
                opacity=opacity
            )
        )

    # Linha de alerta
    fig1.add_hline(y=0.55, line_dash="dash", line_color="#F0A500",
                   annotation_text="Limiar de Atenção (0.55)", annotation_position="top left")

    fig1.update_layout(
        title="<b>BLOCO 1:</b> Evolução do Score de Anomalia (Frota Completa)",
        height=400, template="plotly_dark", paper_bgcolor="#0D1117", plot_bgcolor="#161B22",
        margin=dict(l=40, r=40, t=60, b=40),
        xaxis_title="Tempo", yaxis_title="Score de Risco"
    )
    fig1.update_xaxes(showgrid=True, gridcolor='#2A3444')
    fig1.update_yaxes(showgrid=True, gridcolor='#2A3444')
    fig1.show()


    if not df_alertas.empty:
        # =====================================================================
        # BLOCO 2: Volume de Alertas por Máquina
        # =====================================================================
        alertas_por_ativo = df_alertas['ativo'].value_counts().reset_index()
        alertas_por_ativo.columns = ['ativo', 'contagem']

        fig2 = go.Figure(data=[
            go.Bar(
                x=alertas_por_ativo['ativo'], y=alertas_por_ativo['contagem'],
                marker_color=[cores_turbinas.get(a, '#F0A500') for a in alertas_por_ativo['ativo']],
                text=alertas_por_ativo['contagem'], textposition='auto'
            )
        ])

        fig2.update_layout(
            title="<b>BLOCO 2:</b> Máquinas em Estado de Alerta (Volume de Ocorrências)",
            height=400, template="plotly_dark", paper_bgcolor="#0D1117", plot_bgcolor="#161B22",
            margin=dict(l=40, r=40, t=60, b=40),
            xaxis_title="Turbina", yaxis_title="Qtd. de Alertas Emitidos"
        )
        fig2.update_xaxes(showgrid=False)
        fig2.update_yaxes(showgrid=True, gridcolor='#2A3444')
        fig2.show()


        # =====================================================================
        # BLOCO 3: Distribuição de Severidade
        # =====================================================================
        sev_counts = df_alertas['severidade'].value_counts().reset_index()
        sev_counts.columns = ['severidade', 'contagem']

        fig3 = go.Figure(data=[
            go.Pie(
                labels=sev_counts['severidade'], values=sev_counts['contagem'],
                marker=dict(colors=[cores_sev.get(s, '#ffffff') for s in sev_counts['severidade']]),
                hole=0.45, textinfo='percent+value+label'
            )
        ])

        fig3.update_layout(
            title="<b>BLOCO 3:</b> Distribuição de Severidade Global",
            height=400, template="plotly_dark", paper_bgcolor="#0D1117", plot_bgcolor="#161B22",
            margin=dict(l=40, r=40, t=60, b=40)
        )
        fig3.show()

    else:
        print("\n✅ Nenhum alerta detectado nos dados. Blocos 2 e 3 foram suprimidos.")


# --- EXECUÇÃO DOS BLOCOS VISUAIS ---
base_dir = '/content/drive/MyDrive/Colab Notebooks/Portifólio de Projetos/vigil_artefatos/'
out_dir = os.path.join(base_dir, 'outputs')

gerar_visualizacoes_separadas(out_dir)


📊 GERANDO RELATÓRIO VISUAL VIGIL ENERGY...



In [8]:
!pip install -q kaleido==0.2.1

In [9]:
"""
Vigil Energy — 03 · Vigil Dashboard (Estático + Exportação de Artefatos)
========================================================================
Gera a análise visual do motor preditivo em blocos separados e exporta
os artefatos gerados (imagens em alta resolução) para a pasta de outputs.
"""

import os
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# Verificação e instalação do Kaleido (necessário para exportar PNG no Plotly)
try:
    import kaleido
except ImportError:
    print("⏳ Instalando biblioteca 'kaleido' para exportação de imagens...")
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "kaleido"])
    print("✅ Kaleido instalado com sucesso!\n")

# Configuração CRÍTICA: Garante que o Plotly renderize corretamente no Colab
pio.renderers.default = 'colab'

def gerar_dashboard_e_exportar(out_dir: str):
    scores_file = os.path.join(out_dir, "scores_por_ativo.csv")
    alertas_file = os.path.join(out_dir, "alertas_detectados.csv")

    if not os.path.exists(scores_file):
        print("⚠️ Arquivo de scores não encontrado. Execute o modelo Vigil Detect primeiro.")
        return

    # 1. Carga dos Dados
    df_scores = pd.read_csv(scores_file)
    df_scores['timestamp'] = pd.to_datetime(df_scores['timestamp'])

    df_alertas = pd.DataFrame()
    if os.path.exists(alertas_file):
        df_alertas = pd.read_csv(alertas_file)
        df_alertas['timestamp'] = pd.to_datetime(df_alertas['timestamp'])

    # Paleta de cores padronizada (identidade visual)
    cores_turbinas = {
        "T-01": "#1f77b4", "T-02": "#9467bd", "T-03": "#2ca02c",
        "T-04": "#FF4B4B", "T-05": "#e377c2", "T-06": "#8c564b"
    }
    cores_sev = {'CRITICO': '#FF4B4B', 'ALTO': '#F0A500', 'MEDIO': '#3DD68C'}

    print("📊 GERANDO E EXPORTANDO ARTEFATOS VISUAIS VIGIL ENERGY...\n")

    # =====================================================================
    # BLOCO 1: Visão Geral - Evolução do Score de Anomalia
    # =====================================================================
    fig1 = go.Figure()

    for ativo in sorted(df_scores['ativo'].unique()):
        df_plot = df_scores[df_scores['ativo'] == ativo]

        width = 2.5 if ativo == "T-04" else 1.0
        opacity = 1.0 if ativo == "T-04" else 0.6

        fig1.add_trace(
            go.Scatter(
                x=df_plot['timestamp'], y=df_plot['score_anomalia'],
                name=ativo, mode='lines',
                line=dict(color=cores_turbinas.get(ativo, "#ffffff"), width=width),
                opacity=opacity
            )
        )

    fig1.add_hline(y=0.55, line_dash="dash", line_color="#F0A500",
                   annotation_text="Limiar de Atenção (0.55)", annotation_position="top left")

    fig1.update_layout(
        title="<b>BLOCO 1:</b> Evolução do Score de Anomalia Preditiva (Frota Completa)",
        height=450, template="plotly_dark", paper_bgcolor="#0D1117", plot_bgcolor="#161B22",
        margin=dict(l=40, r=40, t=60, b=40),
        xaxis_title="Tempo", yaxis_title="Score de Risco Combinado (0 a 1)"
    )
    fig1.update_xaxes(showgrid=True, gridcolor='#2A3444')
    fig1.update_yaxes(showgrid=True, gridcolor='#2A3444')

    # Exibe na tela e exporta
    fig1.show(renderer="colab")
    path_fig1 = os.path.join(out_dir, "01_evolucao_score_anomalia.png")
    fig1.write_image(path_fig1, width=1200, height=600, scale=2)
    print(f"📁 Exportado: {path_fig1}")


    if not df_alertas.empty:
        # =====================================================================
        # BLOCO 2: Volume de Alertas por Máquina
        # =====================================================================
        alertas_por_ativo = df_alertas['ativo'].value_counts().reset_index()
        alertas_por_ativo.columns = ['ativo', 'contagem']

        fig2 = go.Figure(data=[
            go.Bar(
                x=alertas_por_ativo['ativo'], y=alertas_por_ativo['contagem'],
                marker_color=[cores_turbinas.get(a, '#F0A500') for a in alertas_por_ativo['ativo']],
                text=alertas_por_ativo['contagem'], textposition='auto'
            )
        ])

        fig2.update_layout(
            title="<b>BLOCO 2:</b> Máquinas em Estado de Alerta (Volume de Ocorrências)",
            height=400, template="plotly_dark", paper_bgcolor="#0D1117", plot_bgcolor="#161B22",
            margin=dict(l=40, r=40, t=60, b=40),
            xaxis_title="Ativo", yaxis_title="Quantidade de Alertas Validados"
        )
        fig2.update_xaxes(showgrid=False)
        fig2.update_yaxes(showgrid=True, gridcolor='#2A3444')

        # Exibe na tela e exporta
        fig2.show(renderer="colab")
        path_fig2 = os.path.join(out_dir, "02_volume_alertas_maquina.png")
        fig2.write_image(path_fig2, width=1000, height=500, scale=2)
        print(f"📁 Exportado: {path_fig2}")


        # =====================================================================
        # BLOCO 3: Distribuição de Severidade
        # =====================================================================
        sev_counts = df_alertas['severidade'].value_counts().reset_index()
        sev_counts.columns = ['severidade', 'contagem']

        fig3 = go.Figure(data=[
            go.Pie(
                labels=sev_counts['severidade'], values=sev_counts['contagem'],
                marker=dict(colors=[cores_sev.get(s, '#ffffff') for s in sev_counts['severidade']]),
                hole=0.45, textinfo='percent+value+label'
            )
        ])

        fig3.update_layout(
            title="<b>BLOCO 3:</b> Distribuição da Severidade Global dos Alertas",
            height=400, template="plotly_dark", paper_bgcolor="#0D1117", plot_bgcolor="#161B22",
            margin=dict(l=40, r=40, t=60, b=40)
        )

        # Exibe na tela e exporta
        fig3.show(renderer="colab")
        path_fig3 = os.path.join(out_dir, "03_distribuicao_severidade.png")
        fig3.write_image(path_fig3, width=800, height=500, scale=2)
        print(f"📁 Exportado: {path_fig3}\n")

    else:
        print("\n✅ Nenhum alerta detectado nos dados. Blocos 2 e 3 não gerados.")

    print("=" * 65)
    print("✅ Todos os gráficos foram salvos em alta resolução na sua pasta 'outputs'!")
    print("=" * 65)


# --- DISPARO DO DASHBOARD E EXPORTAÇÃO ---
base_dir = '/content/drive/MyDrive/Colab Notebooks/Portifólio de Projetos/vigil_artefatos/'
out_dir = os.path.join(base_dir, 'outputs')

gerar_dashboard_e_exportar(out_dir)

📊 GERANDO E EXPORTANDO ARTEFATOS VISUAIS VIGIL ENERGY...



📁 Exportado: /content/drive/MyDrive/Colab Notebooks/Portifólio de Projetos/vigil_artefatos/outputs/01_evolucao_score_anomalia.png


📁 Exportado: /content/drive/MyDrive/Colab Notebooks/Portifólio de Projetos/vigil_artefatos/outputs/02_volume_alertas_maquina.png


📁 Exportado: /content/drive/MyDrive/Colab Notebooks/Portifólio de Projetos/vigil_artefatos/outputs/03_distribuicao_severidade.png

✅ Todos os gráficos foram salvos em alta resolução na sua pasta 'outputs'!
